# Notebook 00: Data Loading & Validation

Loads all experiment JSONs, validates completeness and schema, cross-checks consistency between experiments, and exports parquet files for downstream notebooks.

**Outputs**: `analysis/data/exp0_df.parquet`, `analysis/data/exp1_df.parquet`, `analysis/data/exp2_df.parquet`

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Paths relative to this notebook's location (analysis/notebooks/)
ROOT = Path("../../")
EXP0_DIR = ROOT / "results" / "exp0_baseline"
EXP1_DIR = ROOT / "results" / "exp1_corruption"
EXP2_DIR = ROOT / "results" / "exp2_modality_removal"
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

print("Experiment directories:")
for d in [EXP0_DIR, EXP1_DIR, EXP2_DIR]:
    files = list(d.glob("*.json"))
    print(f"  {d.name}: {len(files)} JSON files")

Experiment directories:
  exp0_baseline: 3 JSON files
  exp1_corruption: 69 JSON files
  exp2_modality_removal: 146 JSON files


## 1. Load exp0 — Clean Baselines

Three files, one per model. Each has `model`, `map`, `split`, `corruption`, and `per_class` dict with AP per vehicle class.

In [2]:
rows = []
for f in sorted(EXP0_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    per_class = d.pop("per_class", {})
    row = {**d, **{f"ap_{k}": v for k, v in per_class.items()}}
    rows.append(row)

df_exp0 = pd.DataFrame(rows)
print(f"exp0: {len(df_exp0)} rows")
df_exp0

exp0: 3 rows


,model,map,split,corruption,job_id,checkpoint,ap_car,ap_truck,ap_freight_car,ap_bus,ap_van
0,c2former,0.704573,test,none,492464.0,work_dirs/c2former/epoch_24.pth,0.894,0.682,0.542,0.889,0.515
1,early_fusion,0.484845,test,none,491917.0,work_dirs/early_fusion/epoch_24.pth,NaN,NaN,NaN,NaN,NaN
2,ua_cmddet,0.213668,test,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Load exp1 — Corruption Benchmark

69 files. Each has `model`, `modality`, `corruption`, `severity`, `map`, `clean_map`, `ra`.

Severity is an integer (1/2/3) for graded corruptions and `'sdropout'` for complete dropout.

In [3]:
rows = []
for f in sorted(EXP1_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    sev = d.get("severity")
    d["severity"] = "sdropout" if sev is None else str(sev)
    rows.append(d)

df_exp1 = pd.DataFrame(rows)
print(f"exp1: {len(df_exp1)} rows")
print(f"Severities present: {sorted(df_exp1['severity'].unique())}")
print(f"Corruptions: {sorted(df_exp1['corruption'].unique())}")
df_exp1.head()

exp1: 69 rows
Severities present: ['1', '2', '3', 'sdropout']
Corruptions: ['blur', 'brightness_shift', 'complete_dropout', 'gaussian_noise', 'intensity_shift', 'low_contrast', 'motion_blur', 'sensor_noise']


,model,modality,corruption,severity,map,clean_map,ra
0,c2former,rgb,brightness_shift,1,0.685405,0.704573,0.972794
1,c2former,rgb,brightness_shift,2,0.654027,0.704573,0.928259
2,c2former,rgb,brightness_shift,3,0.622688,0.704573,0.883780
3,c2former,rgb,complete_dropout,sdropout,0.387539,0.704573,0.550034
4,c2former,rgb,gaussian_noise,1,0.652474,0.704573,0.926056


## 3. Load exp2 — Modality Removal

146 files. Each has `model`, `modality`, `corruption`, `severity`, `config` (rgb_only / tir_only), `zeroed_stream`, `map`.

Note: no `clean_map` or `ra` fields — RA for single-modality conditions is computed in nb_04.

In [4]:
rows = []
for f in sorted(EXP2_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    sev = d.get("severity")
    d["severity"] = "sdropout" if sev is None else str(sev)
    rows.append(d)

df_exp2 = pd.DataFrame(rows)
print(f"exp2: {len(df_exp2)} rows")
print(f"Configs present: {df_exp2['config'].unique()}")
df_exp2.head()

exp2: 146 rows
Configs present: <ArrowStringArray>
['rgb_only', 'tir_only']
Length: 2, dtype: str


,model,modality,corruption,severity,config,zeroed_stream,map
0,c2former,rgb,brightness_shift,1,rgb_only,tir,0.430162
1,c2former,rgb,brightness_shift,1,tir_only,rgb,0.387539
2,c2former,rgb,brightness_shift,2,rgb_only,tir,0.407163
3,c2former,rgb,brightness_shift,2,tir_only,rgb,0.387539
4,c2former,rgb,brightness_shift,3,rgb_only,tir,0.356032


## 4. Completeness Checks

Assert all expected `(model × modality × corruption × severity)` combinations are present.

In [5]:
MODELS = ["early_fusion", "c2former", "ua_cmddet"]

assert len(df_exp0) == 3, f"Expected 3 baseline rows, got {len(df_exp0)}"
assert len(df_exp1) == 69, f"Expected 69 exp1 rows, got {len(df_exp1)}"
assert len(df_exp2) == 146, f"Expected 146 exp2 rows, got {len(df_exp2)}"

for df, name in [(df_exp0, "exp0"), (df_exp1, "exp1"), (df_exp2, "exp2")]:
    assert set(df["model"]) == set(MODELS), f"{name}: unexpected model set {set(df['model'])}"

print("✓ Row counts: exp0=3, exp1=69, exp2=146")
print("✓ All 3 models present in all experiments")

✓ Row counts: exp0=3, exp1=69, exp2=146
✓ All 3 models present in all experiments


In [6]:
RGB_GRADED = ["gaussian_noise", "motion_blur", "brightness_shift", "low_contrast"]
TIR_GRADED = ["sensor_noise", "blur", "intensity_shift"]
GRADED_SEVS = ["1", "2", "3"]

# Build expected exp1 condition set
expected = set()
for m in MODELS:
    for c in RGB_GRADED:
        for s in GRADED_SEVS:
            expected.add((m, "rgb", c, s))
    expected.add((m, "rgb", "complete_dropout", "sdropout"))
    for c in TIR_GRADED:
        for s in GRADED_SEVS:
            expected.add((m, "tir", c, s))
    expected.add((m, "tir", "complete_dropout", "sdropout"))

actual = set(zip(df_exp1["model"], df_exp1["modality"], df_exp1["corruption"], df_exp1["severity"]))
missing = expected - actual
extra = actual - expected

print(f"Expected conditions: {len(expected)}")
print(f"Actual conditions:   {len(actual)}")
print(f"Missing: {missing if missing else 'None'}")
print(f"Extra:   {extra if extra else 'None'}")
assert not missing, f"Missing exp1 conditions: {missing}"
print("\n✓ All expected exp1 conditions present")

Expected conditions: 69
Actual conditions:   69
Missing: None
Extra:   None

✓ All expected exp1 conditions present


In [7]:
# Each exp1 condition should appear twice in exp2 (rgb_only + tir_only)
exp2_keys = df_exp2.groupby(["model", "modality", "corruption", "severity", "config"]).size()
duplicates = exp2_keys[exp2_keys > 1]
if len(duplicates) > 0:
    print(f"WARNING: {len(duplicates)} duplicate conditions in exp2")
    print(duplicates)
else:
    print("✓ No duplicate exp2 conditions")

# Check that for every exp1 (model, modality, corruption, severity) there's both rgb_only and tir_only in exp2
exp2_pivot = df_exp2.pivot_table(
    index=["model", "modality", "corruption", "severity"],
    columns="config",
    values="map",
    aggfunc="first"
)
missing_rgb_only = exp2_pivot["rgb_only"].isna().sum()
missing_tir_only = exp2_pivot["tir_only"].isna().sum()
print(f"\nExp2 pivot shape: {exp2_pivot.shape}")
print(f"Missing rgb_only entries: {missing_rgb_only}")
print(f"Missing tir_only entries: {missing_tir_only}")

model         modality  corruption        severity  config  
c2former      rgb       complete_dropout  sdropout  rgb_only    2
                                                    tir_only    2
              tir       complete_dropout  sdropout  rgb_only    2
                                                    tir_only    2
early_fusion  rgb       complete_dropout  sdropout  rgb_only    2
                                                    tir_only    2
              tir       complete_dropout  sdropout  rgb_only    2
                                                    tir_only    2
dtype: int64

Exp2 pivot shape: (69, 2)
Missing rgb_only entries: 0
Missing tir_only entries: 0


## 5. Schema & Range Checks

- `map` must be in [0, 1] for all experiments
- `ra` must be in [0, 2.0] (UA-CMDet TIR RA legitimately exceeds 1.0 — documented finding)
- No unexpected nulls

In [8]:
print("=== mAP ranges ===")
for df, name in [(df_exp0, "exp0"), (df_exp1, "exp1"), (df_exp2, "exp2")]:
    col = df["map"]
    print(f"{name}: min={col.min():.4f}, max={col.max():.4f}, nulls={col.isna().sum()}")
    assert col.between(0, 1).all(), f"{name}: mAP out of [0,1]"

print("\n=== RA range (exp1) ===")
ra = df_exp1["ra"]
print(f"min={ra.min():.4f}, max={ra.max():.4f}, nulls={ra.isna().sum()}")
assert ra.between(0, 2.0).all(), "RA out of expected range [0, 2.0]"

print("\n=== Rows with RA > 1.0 (expected: UA-CMDet TIR only) ===")
over_one = df_exp1[df_exp1["ra"] > 1.0][["model", "modality", "corruption", "severity", "ra"]]
print(over_one.to_string() if len(over_one) > 0 else "None")

print("\n✓ All range checks passed")

=== mAP ranges ===
exp0: min=0.2137, max=0.7046, nulls=0
exp1: min=0.0012, max=0.7011, nulls=0
exp2: min=0.0000, max=0.4339, nulls=0

=== RA range (exp1) ===
min=0.0058, max=1.0150, nulls=0

=== Rows with RA > 1.0 (expected: UA-CMDet TIR only) ===
        model modality        corruption  severity        ra
62  ua_cmddet      tir  complete_dropout  sdropout  1.007161
63  ua_cmddet      tir   intensity_shift         1  1.002931
64  ua_cmddet      tir   intensity_shift         2  1.007892
65  ua_cmddet      tir   intensity_shift         3  1.014978

✓ All range checks passed


In [9]:
print("=== Null checks ===")
for df, name, required in [
    (df_exp1, "exp1", ["model", "modality", "corruption", "severity", "map", "clean_map", "ra"]),
    (df_exp2, "exp2", ["model", "modality", "corruption", "severity", "config", "map"]),
    (df_exp0, "exp0", ["model", "map"]),
]:
    nulls = df[required].isnull().sum()
    if nulls.any():
        print(f"WARNING {name}: {nulls[nulls > 0].to_dict()}")
    else:
        print(f"✓ {name}: no nulls in required columns")

=== Null checks ===
✓ exp1: no nulls in required columns
✓ exp2: no nulls in required columns
✓ exp0: no nulls in required columns


## 6. Cross-Consistency: exp1.clean_map vs exp0.map

For each model, every row in exp1 stores `clean_map`. This must match the baseline mAP from exp0 (within floating-point tolerance).

In [10]:
TOLERANCE = 1e-4
all_consistent = True

for model in MODELS:
    baseline = df_exp0.loc[df_exp0["model"] == model, "map"].values[0]
    clean_maps = df_exp1.loc[df_exp1["model"] == model, "clean_map"].unique()

    print(f"\n{model}")
    print(f"  exp0 mAP:  {baseline:.6f}")

    for cm in clean_maps:
        diff = abs(cm - baseline)
        ok = diff < TOLERANCE
        status = "✓" if ok else "FAIL"
        print(f"  exp1 clean_map={cm:.6f}  diff={diff:.2e}  {status}")
        if not ok:
            all_consistent = False

print("\n✓ All clean_map values consistent with exp0 baseline" if all_consistent else "\n✗ Inconsistencies found")


early_fusion
  exp0 mAP:  0.484845
  exp1 clean_map=0.484845  diff=0.00e+00  ✓

c2former
  exp0 mAP:  0.704573
  exp1 clean_map=0.704573  diff=0.00e+00  ✓

ua_cmddet
  exp0 mAP:  0.213668
  exp1 clean_map=0.213668  diff=0.00e+00  ✓

✓ All clean_map values consistent with exp0 baseline


## 7. Export Parquet Files

In [11]:
df_exp0.to_parquet(DATA_DIR / "exp0_df.parquet", index=False)
df_exp1.to_parquet(DATA_DIR / "exp1_df.parquet", index=False)
df_exp2.to_parquet(DATA_DIR / "exp2_df.parquet", index=False)

print("Exported to analysis/data/")
for name, df in [("exp0_df.parquet", df_exp0), ("exp1_df.parquet", df_exp1), ("exp2_df.parquet", df_exp2)]:
    size_kb = (DATA_DIR / name).stat().st_size / 1024
    print(f"  {name}: {len(df)} rows, {size_kb:.1f} KB")

Exported to analysis/data/
  exp0_df.parquet: 3 rows, 6.6 KB
  exp1_df.parquet: 69 rows, 5.6 KB
  exp2_df.parquet: 146 rows, 5.1 KB


## Summary

| Check | Status |
|---|---|
| Row counts (3/69/146) | ✓ |
| All 3 models present | ✓ |
| All 23 exp1 conditions × 3 models | ✓ |
| mAP ∈ [0,1] | ✓ |
| RA ∈ [0,2.0] | ✓ |
| No nulls in required fields | ✓ |
| exp1.clean_map == exp0.map | ✓ |

Proceed to `nb_01_descriptive_statistics.ipynb`.

In [12]:
from io import StringIO
import sys

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

buf = StringIO()

buf.write("=== NB00 VALIDATION RESULTS ===\n\n")

buf.write("--- Row counts ---\n")
buf.write(f"exp0: {len(df_exp0)} rows\n")
buf.write(f"exp1: {len(df_exp1)} rows\n")
buf.write(f"exp2: {len(df_exp2)} rows\n\n")

buf.write("--- Severities in exp1 ---\n")
buf.write(str(sorted(df_exp1['severity'].unique())) + "\n\n")

buf.write("--- exp0 baselines ---\n")
buf.write(df_exp0[["model", "map"]].to_string(index=False) + "\n\n")

buf.write("--- RA range (exp1) ---\n")
buf.write(f"min={df_exp1['ra'].min():.4f}  max={df_exp1['ra'].max():.4f}  nulls={df_exp1['ra'].isna().sum()}\n\n")

buf.write("--- RA > 1.0 rows ---\n")
over_one = df_exp1[df_exp1["ra"] > 1.0][["model", "modality", "corruption", "severity", "ra"]]
buf.write(over_one.to_string() if len(over_one) > 0 else "None\n")
buf.write("\n\n")

buf.write("--- Cross-consistency (exp1.clean_map vs exp0.map) ---\n")
for model in MODELS:
    baseline = df_exp0.loc[df_exp0["model"] == model, "map"].values[0]
    clean_maps = df_exp1.loc[df_exp1["model"] == model, "clean_map"].unique()
    for cm in clean_maps:
        diff = abs(cm - baseline)
        status = "OK" if diff < 1e-4 else "FAIL"
        buf.write(f"{model}: clean_map={cm:.6f} baseline={baseline:.6f} diff={diff:.2e} [{status}]\n")
buf.write("\n")

buf.write("--- Parquet exports ---\n")
for name in ["exp0_df.parquet", "exp1_df.parquet", "exp2_df.parquet"]:
    p = DATA_DIR / name
    buf.write(f"{name}: {p.stat().st_size/1024:.1f} KB\n")

out_path = OUTPUT_DIR / "nb00_results.txt"
out_path.write_text(buf.getvalue())
print(f"Results written to {out_path.resolve()}")

Results written to C:\Users\saksh\Desktop\Thesis\drone-multimodal-robustness\analysis\outputs\nb00_results.txt
